# Section 7 · Observability 2 — Local Tracing with Jaeger

Jaeger gives you a visual waterfall of each trace on your own machine.

Start Jaeger in a terminal first:

```bash
docker run -d --name jaeger \
  -e COLLECTOR_OTLP_ENABLED=true \
  -p 16686:16686 -p 4317:4317 -p 4318:4318 \
  jaegertracing/all-in-one:latest
```

Then open http://localhost:16686.

> Each tracing notebook should run in a **fresh kernel**: OpenTelemetry allows only one global tracer provider per Python process.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands_tools import calculator

# Send spans to Jaeger's OTLP/HTTP endpoint
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "http://localhost:4318"
os.environ["OTEL_SERVICE_NAME"] = "strands-agents-demo"

strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_otlp_exporter()

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
agent = Agent(model=model, system_prompt="You are a helpful assistant.", tools=[calculator])

response = agent("Explain the difference between traces, spans, and metrics in OpenTelemetry. Then compute 12% of 9,450.")

# Spans are sent in batches; flush before looking at Jaeger
strands_telemetry.tracer_provider.force_flush()

In Jaeger, choose the **strands-agents-demo** service and click **Find Traces**. The waterfall shows how long each model call took, tokens per call, the tool call with its inputs and output, and the total duration.

Stop Jaeger when you're done: `docker rm -f jaeger`